# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


In [6]:
# 🛠️ TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception:
        return "Error in calculation"

In [7]:
# 🛠️ TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

## 🤖 Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

In [8]:
# 🤖 AGENT FUNCTION (IMPLEMENTED)

def agent(query: str):
    import re
    import urllib.request
    import json
    import urllib.parse
    query_lower = query.lower()
    try:
        # 1. Routing for Calculator Tool
        if "calculate" in query_lower:
            idx = query_lower.find("calculate")
            expression = query[idx + len("calculate"):].strip().rstrip("?. ")
            result = calculator(expression)
            if result == "Error in calculation":
                return {
                    "type": "error",
                    "result": f"Failed to evaluate calculation: '{expression}'"
                }
            return {
                "type": "calculation",
                "result": result
            }

        # 2. Routing for Keyword Extractor Tool
        elif "keywords" in query_lower:
            idx = query_lower.find("keywords")
            remaining = query[idx + len("keywords"):].strip()
            while True:
                prev_len = len(remaining)
                remaining_lower = remaining.lower()
                for prefix in ["from", "of", "in", "for", ":"]:
                    if remaining_lower.startswith(prefix):
                        remaining = remaining[len(prefix):].strip()
                        break
                if len(remaining) == prev_len:
                    break
            
            result = extract_keywords(remaining)
            return {
                "type": "keywords",
                "result": result
            }

        # 3. Routing for General Queries
        else:
            # Attempt to fetch dynamic explanation from Wikipedia API
            wiki_result = None
            try:
                # Extract and clean topic name from query
                topic_clean = query_lower
                for prefix in ["explain me about", "explain about", "explain me", "explain", "what is a", "what is an", "what is", "who is", "define", "tell me about", "tell me"]:
                    if topic_clean.startswith(prefix):
                        topic_clean = topic_clean[len(prefix):].strip()
                        break
                topic_clean = topic_clean.rstrip("?.! ")
                
                # Try Title Case query first (Wikipedia format preference)
                topic_title = topic_clean.title().replace(" ", "_")
                encoded_topic = urllib.parse.quote(topic_title)
                url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{encoded_topic}"
                req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
                with urllib.request.urlopen(req, timeout=4) as response:
                    data = json.loads(response.read().decode('utf-8'))
                    if 'extract' in data and data['extract']:
                        wiki_result = data['extract']
            except Exception:
                # Fallback to exact raw query (lowercase) with underscores
                try:
                    topic_raw = topic_clean.replace(" ", "_")
                    encoded_raw = urllib.parse.quote(topic_raw)
                    url_raw = f"https://en.wikipedia.org/api/rest_v1/page/summary/{encoded_raw}"
                    req = urllib.request.Request(url_raw, headers={'User-Agent': 'Mozilla/5.0'})
                    with urllib.request.urlopen(req, timeout=4) as response:
                        data = json.loads(response.read().decode('utf-8'))
                        if 'extract' in data and data['extract']:
                            wiki_result = data['extract']
                except Exception:
                    pass
            
            if wiki_result:
                return {
                    "type": "general",
                    "result": wiki_result
                }

            # Local Fallback Dictionary (Offline / Backup mode)
            general_responses = {
                "deep learning": "Deep learning is a subset of machine learning based on artificial neural networks with multiple layers (hence 'deep') that learn representations from data.",
                "dl": "DL (Deep Learning) is a subset of machine learning based on artificial neural networks with multiple layers that learn representations from data.",
                "machine learning": "Machine learning is a subset of artificial intelligence (AI) that focuses on building systems that learn from, and make decisions based on, data.",
                "ml": "ML (Machine Learning) is a subset of artificial intelligence (AI) that focuses on building systems that learn from, and make decisions based on, data.",
                "neural network": "Neural networks are computational models inspired by the structure and function of biological brains, used to approximate functions that can depend on a large number of inputs.",
                "ann": "ANN (Artificial Neural Network) is a computational model inspired by biological neural networks, used to estimate or approximate functions.",
                "cnn": "CNN (Convolutional Neural Network) is a class of deep neural networks, most commonly applied to analyzing visual imagery.",
                "rnn": "RNN (Recurrent Neural Network) is a class of artificial neural networks where connections between nodes can form a directed cycle, allowing them to exhibit temporal dynamic behavior.",
                "artificial intelligence": "Artificial intelligence (AI) is the intelligence of machines or software, as opposed to the intelligence of human beings or other animals.",
                "ai": "AI (Artificial Intelligence) is the intelligence of machines or software, as opposed to the intelligence of human beings or other animals.",
                "python": "Python is a high-level, general-purpose, interpreted programming language known for its readability, simplicity, and extensive ecosystem.",
                "agent": "An AI agent is an autonomous entity that perceives its environment through sensors, makes decisions using a reasoning engine, and acts upon the environment using tools.",
                "epoch": "An epoch refers to one complete pass of the entire training dataset through the neural network during training.",
                "batch size": "Batch size is the number of training samples processed in one forward and backward pass before the model weights are updated.",
                "learning rate": "The learning rate is a tuning parameter in an optimization algorithm that determines the step size at each iteration while moving toward a minimum of a loss function.",
                "overfitting": "Overfitting occurs when a machine learning model learns the noise in the training data to the extent that it negatively impacts the performance of the model on new data.",
                "overfit": "Overfitting occurs when a machine learning model learns the noise in the training data to the extent that it negatively impacts the performance of the model on new data.",
                "underfitting": "Underfitting occurs when a machine learning model cannot capture the underlying trend of the data, resulting in poor performance on both training and test data.",
                "underfit": "Underfitting occurs when a machine learning model cannot capture the underlying trend of the data, resulting in poor performance on both training and test data.",
                "early stop": "Early stopping is a regularization method used to prevent overfitting by ending model training once performance on a validation dataset stops improving or begins to worsen.",
                "optimizer": "An optimizer is an algorithm or method used to change the attributes of a neural network, such as weights and learning rate, to reduce the loss.",
                "loss function": "A loss function is a method of evaluating how well a specific algorithm models the given data by measuring the difference between predicted and actual values.",
                "activation function": "An activation function is a mathematical formula applied to a neural network node to determine whether it should be activated (fire) or not based on the input.",
                "linear regression": "Linear regression is a supervised machine learning algorithm used to model the relationship between a dependent scalar variable and one or more explanatory variables by fitting a linear equation to observed data.",
                "logistic regression": "Logistic regression is a supervised learning classification algorithm used to predict the probability of a target variable (binary class, e.g. yes/no).",
                "regression": "Regression is a statistical method used to determine the strength and character of the relationship between one dependent variable and a series of other variables.",
                "random forest": "Random Forest is an ensemble machine learning algorithm that constructs a multitude of decision trees at training time and outputs the class or mean prediction of the individual trees.",
                "decision tree": "A decision tree is a non-parametric supervised learning method used for classification and regression tasks, structured in a flowchart-like tree structure.",
                "how are you": "I am doing great, thank you! I am ready to help you with math calculations, keyword extraction, or machine learning queries.",
                "how r u": "I am doing great, thank you! I am ready to help you with math calculations, keyword extraction, or machine learning queries.",
                "hello": "Hello! How can I help you today? You can ask me to calculate something, extract keywords, or explain machine learning concepts.",
                "hi": "Hello! How can I help you today? You can ask me to calculate something, extract keywords, or explain machine learning concepts.",
                "hey": "Hello! How can I help you today? You can ask me to calculate something, extract keywords, or explain machine learning concepts."
            }
            
            result = None
            for key, val in general_responses.items():
                if len(key) <= 3:
                    if re.search(r'\b' + re.escape(key) + r'\b', query_lower):
                        result = val
                        break
                else: 
                    if key in query_lower:
                        result = val
                        break
            
            if not result:
                result = f"Direct response to your query: {query}"
                
            return {
                "type": "general",
                "result": result
            }

    except Exception as e:
        return {
            "type": "error",
            "result": str(e)
        }


## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [11]:
# 🧪 Test Cases

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?"
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

Query: Calculate 20 + 5
Response: {'type': 'calculation', 'result': '25'}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['artificial', 'transforming', 'industries', 'intelligence']}
--------------------------------------------------
Query: What is machine learning?
Response: {'type': 'general', 'result': 'Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without being explicitly programmed. Advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.'}
--------------------------------------------------


In [12]:
# 🎯 Interactive Mode

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))

Enter query (type 'exit' to stop):  explain about deep learning


Response: {'type': 'general', 'result': 'In machine learning, deep learning (DL) focuses on utilizing multilayered neural networks to perform tasks such as classification, regression, and representation learning. The field takes inspiration from biological neuroscience and revolves around stacking artificial neurons into layers and "training" them to process data. The adjective "deep" refers to the use of multiple layers in the network. Methods used can be supervised, semi-supervised or unsupervised.'}


Enter query (type 'exit' to stop):  calculate 15*5


Response: {'type': 'calculation', 'result': '75'}


Enter query (type 'exit' to stop):  exit
